# Custom Metrics: Building Domain-Specific Evaluation Criteria

**Goal:** Learn how to create your own evaluation metrics tailored to your specific use case.

In this tutorial, you'll learn:
1. The anatomy of a metric class
2. How to write effective evaluation prompts
3. Creating metrics for different domains (legal, medical, technical)
4. Advanced prompt engineering for judges
5. Combining multiple custom metrics
6. Best practices for metric design

---

## Why Custom Metrics?

While built-in metrics like Groundedness and Hallucination are useful, your specific domain may require:
- **Industry-specific quality checks** (e.g., medical accuracy, legal precision)
- **Brand voice alignment** (tone, style, formality)
- **Compliance requirements** (regulatory language, disclaimers)
- **User experience factors** (clarity, helpfulness, empathy)
- **Technical accuracy** (code correctness, API usage)

Custom metrics let you encode your domain expertise into the evaluation process.

---

## Setup

In [ ]:
# API Configuration
import os

GROQ_API_KEY = "gsk_YOUR_API_KEY_HERE"

# Or from environment
# GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [2]:
# Imports
from langchain_groq import ChatGroq

from llm_jury.core.evaluator import JuryEvaluator
from llm_jury.judges.llm_judge import LLMJudge
from llm_jury.metrics.base import Metric
from llm_jury.metrics.predefined import GroundednessMetric
from llm_jury.strategies.consensus import MajorityVoting

import pandas as pd
from typing import Any, Dict
from IPython.display import display

print("All imports successful!")

All imports successful!


## Understanding the Metric Base Class

Every metric inherits from the `Metric` abstract base class. Let's examine its structure.

In [3]:
print("\nMetric Base Class Structure:")
print("="*80)
print("""
class Metric(ABC):
    def __init__(self, name, description, scale_min=1.0, scale_max=5.0):
        # Define the metric's identity and scoring range
        pass
    
    @abstractmethod
    def get_prompt(self, context=None) -> str:
        # REQUIRED: Return the evaluation instruction for judges
        pass
    
    def normalize(self, score: float) -> float:
        # Converts raw scores to [0, 1] range
        pass
""")

print("\nKey Components:")
print("1. __init__: Define metric name, description, and scale")
print("2. get_prompt(): Generate the evaluation instruction for judges")
print("3. normalize(): Convert scores to unified range (handled automatically)")
print("\nYou only need to implement get_prompt() - everything else is inherited!")


Metric Base Class Structure:

class Metric(ABC):
    def __init__(self, name, description, scale_min=1.0, scale_max=5.0):
        # Define the metric's identity and scoring range
        pass

    @abstractmethod
    def get_prompt(self, context=None) -> str:
        # REQUIRED: Return the evaluation instruction for judges
        pass

    def normalize(self, score: float) -> float:
        # Converts raw scores to [0, 1] range
        pass


Key Components:
1. __init__: Define metric name, description, and scale
2. get_prompt(): Generate the evaluation instruction for judges
3. normalize(): Convert scores to unified range (handled automatically)

You only need to implement get_prompt() - everything else is inherited!


## Example 1: Simple Custom Metric - Tone Detection

Let's create a metric that evaluates whether text maintains a professional tone.

In [4]:
class ProfessionalToneMetric(Metric):
    """
    Evaluates whether the text maintains a professional, business-appropriate tone.
    Useful for customer service, corporate communications, or business writing.
    """
    
    def __init__(self):
        super().__init__(
            name="ProfessionalTone",
            description="Measures professional tone and appropriate business language",
            scale_min=1.0,
            scale_max=5.0
        )
    
    def get_prompt(self, context: Any = None) -> str:
        """
        Generates the evaluation prompt for judges.
        """
        # Extract the output text to evaluate
        output = context.get("output_text", "") if isinstance(context, dict) else ""
        
        return f"""
You are a Professional Tone Evaluator. Your task is to assess whether the text 
maintains an appropriate professional tone for business communication.

--- TEXT TO EVALUATE ---
{output}

--- EVALUATION CRITERIA ---
Consider the following aspects:
1. Language formality (avoids slang, casual expressions)
2. Emotional restraint (measured, not overly emotional)
3. Respectful phrasing (courteous, non-confrontational)
4. Clear communication (concise, unambiguous)
5. Appropriate vocabulary (business-suitable words)

--- SCORING GUIDE ---
5 = Exemplary professional tone throughout
4 = Mostly professional with minor lapses
3 = Mixed tone, some unprofessional elements
2 = Largely unprofessional tone
1 = Completely inappropriate for business context

Provide your score (1-5) and explain your reasoning.

Score: <number>
Reasoning: <explanation>
"""

# Create instance
tone_metric = ProfessionalToneMetric()
print(f"Created metric: {tone_metric.name}")
print(f"Description: {tone_metric.description}")
print(f"Scale: {tone_metric.scale_min} to {tone_metric.scale_max}")

Created metric: ProfessionalTone
Description: Measures professional tone and appropriate business language
Scale: 1.0 to 5.0


## Test the Tone Metric

Let's evaluate two different customer service responses: one professional, one casual.

In [5]:
# Initialize judges
judges = [
    LLMJudge(
        model=ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.0, api_key=GROQ_API_KEY),
        name="Llama-3.3-70B"
    ),
    LLMJudge(
        model=ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.0, api_key=GROQ_API_KEY),
        name="Openai-GPT-120B"
    )
]

jury = JuryEvaluator(judges=judges, strategy=MajorityVoting())

# Test Case 1: Professional response
professional_response = """
Thank you for contacting our support team. I understand you're experiencing 
difficulties with your account access. I would be happy to assist you in 
resolving this issue. Please provide your account email address, and I will 
initiate the password reset process immediately.
"""

context_pro = {"output_text": professional_response}
result_pro = jury.evaluate(context_pro, professional_response, tone_metric)

print("\nTest Case 1: Professional Response")
print("="*80)
print(f"Score: {result_pro.final_score}/5")
print(f"Valid: {result_pro.is_valid}")
print(f"\nJudge Reasoning:")
for score in result_pro.manifest.individual_scores:
    print(f"\n{score.judge_id}: {score.score}/5")
    print(f"{score.reasoning[:200]}...")


Test Case 1: Professional Response
Score: 5.0/5
Valid: True

Judge Reasoning:

Llama-3.3-70B: 5.0/5
The text maintains an exemplary professional tone throughout. It uses formal language, avoiding slang and casual expressions. The tone is emotionally restrained, measured, and respectful, with courteo...

Openai-GPT-120B: 5.0/5
The text demonstrates a highly professional tone suitable for business communication. It uses formal language without slang, maintains emotional restraint by focusing on the issue rather than expressi...


In [6]:
# Test Case 2: Casual response
casual_response = """
Hey there! Ugh, account issues are the worst, right? No worries tho, we got you! 
Just shoot me your email and I'll fix that password thing ASAP. You'll be back 
in no time, promise!
"""

context_casual = {"output_text": casual_response}
result_casual = jury.evaluate(context_casual, casual_response, tone_metric)

print("\nTest Case 2: Casual Response")
print("="*80)
print(f"Score: {result_casual.final_score}/5")
print(f"Valid: {result_casual.is_valid}")
print(f"\nJudge Reasoning:")
for score in result_casual.manifest.individual_scores:
    print(f"\n{score.judge_id}: {score.score}/5")
    print(f"{score.reasoning[:200]}...")


Test Case 2: Casual Response
Score: 2.0/5
Valid: False

Judge Reasoning:

Llama-3.3-70B: 2.0/5
The text lacks a professional tone in several aspects. It starts with an informal greeting ("Hey there!") and uses casual expressions ("Ugh", "No worries tho", "we got you", "shoot me your email", "pa...

Openai-GPT-120B: 2.0/5
The text uses several informal and slang expressions (“Hey there!”, “Ugh”, “tho”, “shoot me your email”, “ASAP”) which lower its language formality and make the vocabulary unsuitable for a business se...


## Example 2: Domain-Specific Metric - Medical Accuracy

For healthcare applications, we need to evaluate medical information accuracy and safety.

In [7]:
class MedicalAccuracyMetric(Metric):
    """
    Evaluates medical information for accuracy, appropriate disclaimers, and safety.
    Critical for healthcare chatbots and medical information systems.
    """
    
    def __init__(self):
        super().__init__(
            name="MedicalAccuracy",
            description="Evaluates medical information accuracy and safety",
            scale_min=1.0,
            scale_max=5.0
        )
    
    def get_prompt(self, context: Any = None) -> str:
        source = context.get("source_text", "") if isinstance(context, dict) else ""
        output = context.get("output_text", "") if isinstance(context, dict) else ""
        
        return f"""
You are a Medical Information Safety Evaluator. Assess the medical accuracy and 
safety of the response based on the provided medical reference.

--- MEDICAL REFERENCE (Trusted Source) ---
{source}

--- RESPONSE TO EVALUATE ---
{output}

--- CRITICAL EVALUATION CRITERIA ---

1. ACCURACY: Does the response accurately reflect the medical information?
   - No contradictions with the source
   - No fabricated medical facts
   - Dosages, symptoms, and treatments match source

2. SAFETY: Does the response include necessary safety information?
   - Appropriate disclaimers (e.g., "consult a doctor")
   - Warning about serious symptoms
   - No dangerous advice or omissions

3. CLARITY: Is medical terminology used appropriately?
   - Avoids unnecessary jargon
   - Explains complex terms when used
   - Clear and unambiguous

4. COMPLETENESS: Are critical details included?
   - Side effects mentioned if relevant
   - Contraindications noted
   - Important warnings present

--- SCORING GUIDE ---
5 = Medically accurate, safe, with all necessary disclaimers
4 = Accurate and safe, minor clarity issues
3 = Mostly accurate, missing some safety elements
2 = Contains inaccuracies or safety concerns
1 = Dangerous or severely inaccurate medical information

IMPORTANT: Be very strict. Any safety concern should result in a low score.

Score: <number>
Reasoning: <detailed explanation of medical safety assessment>
"""

medical_metric = MedicalAccuracyMetric()
print(f"Created metric: {medical_metric.name}")

Created metric: MedicalAccuracy


In [8]:
# Test the medical metric
medical_source = """
Ibuprofen is a nonsteroidal anti-inflammatory drug (NSAID) used to reduce fever and 
treat pain or inflammation. The recommended adult dose is 200-400mg every 4-6 hours, 
not exceeding 1200mg per day without medical supervision. Common side effects include 
stomach upset and nausea. Serious side effects include stomach bleeding and kidney problems. 
Avoid if you have a history of stomach ulcers or kidney disease. Always take with food or milk.
"""

good_medical_response = """
Ibuprofen can help with your pain. The typical adult dose is 200-400mg every 4-6 hours, 
with a maximum of 1200mg daily unless directed otherwise by your doctor. Take it with food 
to minimize stomach upset. You should not use ibuprofen if you have stomach ulcers or kidney 
problems. If you experience stomach pain or notice dark stools, stop taking it and contact 
your healthcare provider immediately. This is general information - please consult your 
doctor for personalized medical advice.
"""

medical_context = {
    "source_text": medical_source,
    "output_text": good_medical_response
}

result_medical = jury.evaluate(medical_context, good_medical_response, medical_metric)

print("\nMedical Response Evaluation")
print("="*80)
print(f"Score: {result_medical.final_score}/5")
print(f"Valid: {result_medical.is_valid}")
print(f"\nJudge Assessment:")
print(result_medical.manifest.individual_scores[0].reasoning)


Medical Response Evaluation
Score: 5.0/5
Valid: True

Judge Assessment:
The response accurately reflects the medical information provided in the source text, with no contradictions or fabricated medical facts. The dosages, symptoms, and treatments mentioned match the source. The response includes necessary safety information, such as taking the medication with food to minimize stomach upset, warning about serious symptoms like stomach pain and dark stools, and advising to stop taking the medication and contact a healthcare provider if these symptoms occur. The response also includes appropriate disclaimers, such as consulting a doctor for personalized medical advice. The medical terminology used is clear and unambiguous, avoiding unnecessary jargon. Critical details, including side effects, contraindications, and important warnings, are included. Overall, the response is medically accurate, safe, and complete, with all necessary disclaimers.


## Example 3: Advanced Metric - Code Quality

For technical documentation or code generation, we need to evaluate code correctness.

In [11]:
class CodeQualityMetric(Metric):
    """
    Evaluates code snippets for correctness, best practices, and clarity.
    Useful for code generation, technical documentation, and API examples.
    """
    
    def __init__(self):
        super().__init__(
            name="CodeQuality",
            description="Evaluates code correctness, best practices, and readability",
            scale_min=1.0,
            scale_max=5.0
        )
    
    def get_prompt(self, context: Any = None) -> str:
        task = context.get("task_description", "") if isinstance(context, dict) else ""
        code = context.get("output_text", "") if isinstance(context, dict) else ""
        language = context.get("language", "Python") if isinstance(context, dict) else "Python"
        
        return f"""
You are a Code Quality Evaluator. Assess the quality of this {language} code snippet.

--- TASK DESCRIPTION ---
{task}

--- CODE TO EVALUATE ---
{code}

--- EVALUATION CRITERIA ---
Provide a single score based on the following criteria:
1. CORRECTNESS (40%):
   - Does the code accomplish the task?
   - Are there syntax errors?
   - Does it handle edge cases?
   - Are there logical errors?

2. BEST PRACTICES (30%):
   - Follows {language} conventions
   - Proper error handling
   - Efficient algorithms
   - No security vulnerabilities

3. READABILITY (20%):
   - Clear variable names
   - Appropriate comments
   - Logical structure
   - Consistent formatting

4. MAINTAINABILITY (10%):
   - Modular design
   - Reusable components
   - Easy to modify

--- SCORING GUIDE ---
5 = Production-ready code with excellent practices
4 = Good code, minor improvements possible
3 = Works but has notable issues
2 = Significant problems or bad practices
1 = Incorrect, unsafe, or unusable code

Score: <number>
Reasoning: <detailed technical assessment>
"""

code_metric = CodeQualityMetric()
print(f"Created metric: {code_metric.name}")

Created metric: CodeQuality


In [12]:
# Test the code quality metric
code_context = {
    'task_description': 'Write a function to validate email addresses',
    'language': 'Python',
    'output_text': '''import re

def validate_email(email):
    """
    Validates an email address using regex.
    
    Args:
        email (str): Email address to validate
    
    Returns:
        bool: True if valid, False otherwise
    """
    if not email or not isinstance(email, str):
        return False
    
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))
'''
}

result_code = jury.evaluate(code_context, code_context['output_text'], code_metric)

print('\\nCode Quality Evaluation')
print('='*80)
print(f'Score: {result_code.final_score}/5')
print(f'Valid: {result_code.is_valid}')
print(f'\\nJudge Assessment:')
print(result_code.manifest.individual_scores[0].reasoning)

\nCode Quality Evaluation
Score: 4.0/5
Valid: True
\nJudge Assessment:
The provided Python code snippet is well-structured and effectively validates email addresses using a regular expression. It correctly handles edge cases such as empty strings and non-string inputs. The use of a clear and descriptive docstring enhances readability. The code adheres to Python conventions and is free of syntax errors. However, there are some minor improvements that could be made. For instance, the regular expression pattern could be defined as a constant at the top of the file to improve maintainability. Additionally, the function could be made more robust by handling exceptions that may occur during the execution of the `re.match` function. The code does not have any notable security vulnerabilities. Overall, the code is of good quality, but there is room for minor improvements.


## Example 4: Contextual Metric - Brand Voice Alignment

Ensure generated content matches your brand's style and personality.

In [13]:
class BrandVoiceMetric(Metric):
    """
    Evaluates alignment with specific brand voice guidelines.
    Customizable with brand-specific attributes.
    """
    
    def __init__(self, brand_attributes: Dict[str, str]):
        """
        Args:
            brand_attributes: Dictionary of brand characteristics
                Example: {
                    "tone": "friendly but professional",
                    "personality": "helpful, empowering",
                    "avoid": "jargon, corporate speak",
                    "use": "simple language, active voice"
                }
        """
        super().__init__(
            name="BrandVoice",
            description=f"Evaluates alignment with brand voice",
            scale_min=1.0,
            scale_max=5.0
        )
        self.brand_attributes = brand_attributes
    
    def get_prompt(self, context: Any = None) -> str:
        output = context.get("output_text", "") if isinstance(context, dict) else ""
        
        # Build brand guidelines section
        guidelines = "\n".join([
            f"   {key.upper()}: {value}" 
            for key, value in self.brand_attributes.items()
        ])
        
        return f"""
You are a Brand Voice Evaluator. Assess how well the text aligns with the 
specified brand voice and style guidelines.

--- BRAND GUIDELINES ---
{guidelines}

--- TEXT TO EVALUATE ---
{output}

--- EVALUATION CRITERIA ---

Assess alignment with each brand attribute:
1. Does the tone match the brand personality?
2. Are prohibited elements avoided?
3. Are encouraged elements present?
4. Would a customer recognize this as our brand?
5. Is the voice consistent throughout?

--- SCORING GUIDE ---
5 = Perfect brand alignment, exemplary voice
4 = Strong alignment, minor deviations
3 = Recognizable but inconsistent
2 = Weak alignment, misses key attributes
1 = Completely off-brand

Score: <number>
Reasoning: <specific examples of alignment or misalignment>
"""

# Create a brand voice metric for a friendly tech company
tech_brand = BrandVoiceMetric(
    brand_attributes={
        "tone": "friendly, approachable, never condescending",
        "personality": "helpful guide, not distant expert",
        "avoid": "corporate jargon, overly technical terms without explanation",
        "use": "conversational language, 'we' and 'you', active voice",
        "style": "clear and concise, empowering not overwhelming"
    }
)

print(f"Created metric: {tech_brand.name}")
print(f"\nBrand Attributes:")
for key, value in tech_brand.brand_attributes.items():
    print(f"  {key}: {value}")

Created metric: BrandVoice

Brand Attributes:
  tone: friendly, approachable, never condescending
  personality: helpful guide, not distant expert
  avoid: corporate jargon, overly technical terms without explanation
  use: conversational language, 'we' and 'you', active voice
  style: clear and concise, empowering not overwhelming


In [15]:
# Test brand-aligned vs off-brand responses
on_brand_text = """
We're here to help you get the most out of our platform! Setting up your account 
is simple - just follow these three steps, and you'll be up and running in no time. 
If you get stuck anywhere, our support team is just a click away.
"""

off_brand_text = """
In order to facilitate the initialization of your user credentials within our 
enterprise solution, please adhere to the following procedural guidelines. Failure 
to comply with the specified protocols may result in access denial.
"""

# Evaluate on-brand
on_brand_context = {"output_text": on_brand_text}
result_on = jury.evaluate(on_brand_context, on_brand_text, tech_brand)

print("\nOn-Brand Text Evaluation")
print("="*80)
print(f"Score: {result_on.final_score}/5")
print(f"\nJudge Assessment:")
print(result_on.manifest.individual_scores[0].reasoning[:300])

# Evaluate off-brand
off_brand_context = {"output_text": off_brand_text}
result_off = jury.evaluate(off_brand_context, off_brand_text, tech_brand)

print("\n\nOff-Brand Text Evaluation")
print("="*80)
print(f"Score: {result_off.final_score}/5")
print(f"\nJudge Assessment:")
print(result_off.manifest.individual_scores[0].reasoning[:300])


On-Brand Text Evaluation
Score: 5.0/5

Judge Assessment:
The text fully embodies the brand guidelines. It uses a friendly, approachable tone without any hint of condescension, positioning the writer as a helpful guide (“We’re here to help…”). There is no corporate jargon or unexplained technical language. Conversational phrasing with “we” and “you” is pre


Off-Brand Text Evaluation
Score: 1.0/5

Judge Assessment:
The passage is formal, dense, and uses corporate jargon (“facilitate the initialization,” “procedural guidelines,” “access denial”) that directly violates the brand’s directive to avoid jargon and overly technical language. It lacks the friendly, approachable tone and the conversational elements (“w


## Combining Multiple Custom Metrics

Real-world evaluations often require checking multiple dimensions simultaneously.

In [16]:
# Create a comprehensive evaluation with multiple metrics
customer_response = """
Thank you for reaching out about your billing concern. I understand that 
unexpected charges can be frustrating. Let me help you resolve this.

Looking at your account, I can see the charge in question is for our premium 
features subscription that was activated on March 1st. This is billed monthly 
at $29.99. If this was activated by mistake, I'd be happy to cancel it and 
process a refund for you right away.

Would you like me to proceed with the cancellation and refund?
"""

# Evaluate with multiple metrics
metrics_to_test = [
    tone_metric,
    tech_brand,
    GroundednessMetric()  # Mix custom with built-in
]

# Context with source information
multi_context = {
    "output_text": customer_response,
    "source_text": "Premium subscription: $29.99/month. Activated March 1st. Cancellation and refund available."
}

print("\nMulti-Metric Evaluation of Customer Service Response")
print("="*80)

results = {}
for metric in metrics_to_test:
    result = jury.evaluate(multi_context, customer_response, metric)
    results[metric.name] = result
    print(f"\n{metric.name}: {result.final_score}/5 (Confidence: {result.confidence:.0%})")

# Create summary table
summary_data = [
    {
        'Metric': name,
        'Score': f"{result.final_score}/5",
        'Valid': result.is_valid,
        'Confidence': f"{result.confidence:.0%}"
    }
    for name, result in results.items()
]

summary_df = pd.DataFrame(summary_data)
print("\n\nSummary:")
display(summary_df)


Multi-Metric Evaluation of Customer Service Response

ProfessionalTone: 5.0/5 (Confidence: 100%)

BrandVoice: 4.0/5 (Confidence: 50%)

Groundedness: 4.0/5 (Confidence: 50%)


Summary:


,Metric,Score,Valid,Confidence
0,ProfessionalTone,5.0/5,True,100%
1,BrandVoice,4.0/5,True,50%
2,Groundedness,4.0/5,True,50%


## Best Practices for Custom Metrics

Guidelines for creating effective custom metrics.

BEST PRACTICES FOR CUSTOM METRIC DESIGN
================================================================================

1. CLEAR PROMPT STRUCTURE
   - Start with role definition ("You are a [X] Evaluator")
   - Clearly separate sections (use headers)
   - Provide context, then criteria, then scoring guide
   - End with explicit output format request

2. SPECIFIC EVALUATION CRITERIA
   - List concrete things to check (not vague "quality")
   - Provide examples of good vs. bad
   - Weight criteria if some are more important
   - Make criteria measurable or observable

3. WELL-DEFINED SCORING SCALE
   - Describe what each score means
   - Make levels distinct and non-overlapping
   - Use consistent scale across metrics (typically 1-5)
   - Anchor scores with examples when possible

4. DOMAIN EXPERTISE
   - Encode your domain knowledge into criteria
   - Include safety/compliance requirements
   - Reference industry standards
   - Consider edge cases specific to your domain

5. TESTING AND ITERATION
   - Test on diverse examples (good, bad, edge cases)
   - Compare judge outputs to human judgments
   - Refine prompts based on results
   - Document metric performance and limitations

6. CONTEXT HANDLING
   - Decide what context judges need (source text, task description, etc.)
   - Make context extraction robust (handle missing fields)
   - Don't overload judges with unnecessary information
   - Format context for readability

7. AVOIDING COMMON PITFALLS
   - Don't ask judges to do multiple unrelated tasks
   - Avoid subjective criteria without anchors
   - Don't assume judges have external knowledge
   - Test for prompt injection vulnerabilities

8. COMBINING METRICS
   - Use multiple focused metrics rather than one complex metric
   - Each metric should evaluate one aspect
   - Consider metric dependencies (e.g., accuracy before style)
   - Weight metrics appropriately for final decisions

================================================================================

## Metric Template: Quick Start Guide

A reusable template for creating your own metrics.

In [17]:
template_code = '''
class MyCustomMetric(Metric):
    """
    [Brief description of what this metric evaluates]
    
    Use case: [When to use this metric]
    """
    
    def __init__(self, **kwargs):
        """
        Initialize with any custom parameters needed.
        
        Args:
            **kwargs: Custom configuration options
        """
        super().__init__(
            name="MyMetricName",
            description="Short description for logs/reports",
            scale_min=1.0,  # Adjust as needed
            scale_max=5.0   # Adjust as needed
        )
        # Store any custom parameters
        self.custom_param = kwargs.get('custom_param', 'default')
    
    def get_prompt(self, context: Any = None) -> str:
        """
        Generate the evaluation prompt.
        
        Args:
            context: Dictionary with keys like:
                - output_text: The text to evaluate (required)
                - source_text: Reference material (optional)
                - task_description: What was requested (optional)
                - [any custom keys you need]
        """
        # Extract what you need from context
        output = context.get("output_text", "") if isinstance(context, dict) else ""
        source = context.get("source_text", "") if isinstance(context, dict) else ""
        
        return f"""
You are a [Role Name] Evaluator. [Brief role description].

--- [SECTION 1: CONTEXT] ---
{source}

--- [SECTION 2: CONTENT TO EVALUATE] ---
{output}

--- EVALUATION CRITERIA ---
1. [Criterion 1]: [Specific things to check]
2. [Criterion 2]: [Specific things to check]
3. [Criterion 3]: [Specific things to check]

--- SCORING GUIDE ---
5 = [Best case description]
4 = [Good case description]
3 = [Acceptable case description]
2 = [Poor case description]
1 = [Worst case description]

[Any additional instructions or warnings]

Score: <number>
Reasoning: <explanation>
"""
'''

print("CUSTOM METRIC TEMPLATE")
print("="*80)
print(template_code)
print("\nCopy this template and customize for your use case!")

CUSTOM METRIC TEMPLATE

class MyCustomMetric(Metric):
    """
    [Brief description of what this metric evaluates]

    Use case: [When to use this metric]
    """

    def __init__(self, **kwargs):
        """
        Initialize with any custom parameters needed.

        Args:
            **kwargs: Custom configuration options
        """
        super().__init__(
            name="MyMetricName",
            description="Short description for logs/reports",
            scale_min=1.0,  # Adjust as needed
            scale_max=5.0   # Adjust as needed
        )
        # Store any custom parameters
        self.custom_param = kwargs.get('custom_param', 'default')

    def get_prompt(self, context: Any = None) -> str:
        """
        Generate the evaluation prompt.

        Args:
            context: Dictionary with keys like:
                - output_text: The text to evaluate (required)
                - source_text: Reference material (optional)
                - task_descriptio

## Advanced: Metric with Multiple Scales

Sometimes you need different scoring scales for different aspects.

In [18]:
class ComplianceMetric(Metric):
    """
    Binary compliance check - either compliant or not.
    Uses a 0-1 scale instead of the typical 1-5.
    """
    
    def __init__(self, requirements: list):
        super().__init__(
            name="Compliance",
            description="Binary compliance check",
            scale_min=0.0,  # Not compliant
            scale_max=1.0   # Compliant
        )
        self.requirements = requirements
    
    def get_prompt(self, context: Any = None) -> str:
        output = context.get("output_text", "") if isinstance(context, dict) else ""
        
        requirements_list = "\n".join([
            f"   {i+1}. {req}" for i, req in enumerate(self.requirements)
        ])
        
        return f"""
You are a Compliance Auditor. Check if the text meets ALL required compliance criteria.

--- TEXT TO AUDIT ---
{output}

--- REQUIRED COMPLIANCE CRITERIA ---
{requirements_list}

--- SCORING ---
1.0 = ALL requirements met (fully compliant)
0.0 = ANY requirement missing (non-compliant)

This is a binary check. If even one requirement is missing, score must be 0.0.

Score: <0.0 or 1.0>
Reasoning: <list which requirements are met/missing>
"""

# Example: Financial disclosure compliance
financial_compliance = ComplianceMetric(
    requirements=[
        "Includes risk disclosure statement",
        "States that past performance doesn't guarantee future results",
        "Advises consulting a financial advisor",
        "Clearly labels as not financial advice"
    ]
)

print(f"Created compliance metric with {len(financial_compliance.requirements)} requirements")

Created compliance metric with 4 requirements


## Key Takeaways

### Creating Effective Custom Metrics

**1. Structure Your Prompts Well**
   - Clear role definition
   - Organized sections
   - Specific criteria
   - Explicit scoring guide

**2. Domain Expertise Matters**
   - Encode your knowledge into criteria
   - Include industry-specific requirements
   - Consider safety and compliance
   - Test with domain experts

**3. Test and Iterate**
   - Try on diverse examples
   - Compare to human judgment
   - Refine based on results
   - Document limitations

**4. Combine Metrics Thoughtfully**
   - Each metric evaluates one aspect
   - Use multiple focused metrics
   - Weight appropriately
   - Consider dependencies

**5. Common Use Cases**
   - Brand voice alignment
   - Compliance checking
   - Technical accuracy
   - Domain-specific quality
   - Tone and style

---

**Happy Metric Building!**